# DocsMind lab: choose a BRISKODA embedding model with evidence

This notebook covers: **Chunks → Embed → FAISS Flat search → Post-level retrieval eval**.

It compares `Alibaba-NLP/gte-modernbert-base` and `Qwen/Qwen3-Embedding-0.6B` on the same frozen Superb Mk3 corpus. No password, API key, cloud service, or Hugging Face token is used. Both models are public and run on a DigitalOcean GPU Droplet.


## What this result can and cannot prove

These labels point to posts containing a useful answer, diagnostic procedure, confirmed fix, or—in tutorial threads—the answer in the original post. Unresolved questions are excluded rather than counted as impossible retrieval failures.

In an interview, the real question is not *which model is popular?* It is *which model retrieved your domain evidence better, at what latency, memory, and index cost?*


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import gc
import json
import os
import sys
import time

import faiss
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'docsmind').exists():
    ROOT = ROOT.parent
if not (ROOT / 'docsmind').exists():
    raise RuntimeError('Start Jupyter from the DocsMind repository.')
sys.path.insert(0, str(ROOT))

from docsmind.eval.embedding_retrieval import load_embedding_eval, score_post_rankings
from docsmind.ingestion.briskoda_chunks import BriskodaChunkConfig, build_briskoda_chunks, load_briskoda_posts

LAB_DIR = Path.home() / 'projects/docsmind-data/briskoda/experiments/embedding-answer-v1'
SNAPSHOT_PATH = Path.home() / 'projects/docsmind-data/briskoda/experiments/chunking-v1/snapshot/superb_mk3.briskoda.jsonl'
EVAL_PATH = ROOT / 'data/eval/briskoda_answer_queries.v1.json'
RESULTS_PATH = LAB_DIR / 'benchmark-results.json'
LAB_DIR.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'This lab is intended to run on a DigitalOcean GPU Droplet.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print('Corpus:', SNAPSHOT_PATH)
print('Labels:', EVAL_PATH)


## 1. Rebuild the frozen dense-eligible corpus

The chunk policy stays fixed while models change. Diagnostic chunks marked `index_dense=False` remain available to BM25 later, but they are deliberately excluded from this dense benchmark. Changing chunks and models together would make the comparison impossible to interpret.


In [ ]:
posts = load_briskoda_posts(SNAPSHOT_PATH)
chunk_config = BriskodaChunkConfig()
all_chunks = build_briskoda_chunks(posts, chunk_config)
dense_chunks = [chunk for chunk in all_chunks if chunk['index_dense']]
corpus_texts = [chunk['text'] for chunk in dense_chunks]
corpus_post_ids = np.asarray([chunk['post_id'] for chunk in dense_chunks])

print(f'Source posts: {len(posts):,}')
print(f'All BM25 chunks: {len(all_chunks):,}')
print(f'Dense-eligible chunks: {len(dense_chunks):,}')
print(f'Lexical-only diagnostic chunks: {len(all_chunks) - len(dense_chunks):,}')


## 2. Validate the labelled questions

Every relevant post must exist in the frozen corpus and must have at least one dense-eligible chunk. Otherwise a model could never retrieve it and the benchmark would be invalid.


In [ ]:
eval_dataset = load_embedding_eval(EVAL_PATH)
eval_queries = eval_dataset['queries']
questions = [item['question'] for item in eval_queries]
available_post_ids = set(corpus_post_ids.tolist())
missing_labels = sorted({post_id for item in eval_queries for post_id in item['relevant_post_ids'] if post_id not in available_post_ids})
assert not missing_labels, f'Labels missing from dense corpus: {missing_labels}'
print('Evaluation questions:', len(eval_queries))
print('Label status:', eval_dataset['label_status'])
display(pd.DataFrame(eval_queries)[['id', 'question', 'relevant_post_ids']])


## 3. Lock the candidates and inference settings

GTE is the smaller English encoder: 149M parameters, 768 dimensions, and an 8,192-token context window. Qwen3 is 0.6B parameters, instruction-aware, multilingual, up to 1,024 dimensions, and has a 32K context window.

Qwen's official model card recommends using its stored `query` prompt for retrieval queries, so the benchmark does that. Documents are never given the query instruction. Both candidates produce normalized float32 vectors for cosine search through FAISS inner product.


In [ ]:
MODEL_CONFIGS = [
    {
        'name': 'gte-modernbert-base',
        'model_id': 'Alibaba-NLP/gte-modernbert-base',
        'batch_size': 32,
        'query_prompt_name': None,
        'tokenizer_kwargs': {},
    },
    {
        'name': 'qwen3-embedding-0.6b',
        'model_id': 'Qwen/Qwen3-Embedding-0.6B',
        'batch_size': 8,
        'query_prompt_name': 'query',
        'tokenizer_kwargs': {'padding_side': 'left'},
    },
]
display(pd.DataFrame(MODEL_CONFIGS))


## 4. Embed, search, and measure each candidate

The benchmark embeds the full dense corpus—not a convenient sample. FAISS searches deeper than ten chunks, then collapses duplicate chunks from the same post. This prevents one long post from occupying several result ranks.

`Recall@5/10` asks whether the labelled post is present by that depth. `MRR@10` rewards putting the first relevant post near rank one. Throughput, query latency, peak VRAM, and flat-index size expose the operational cost.


In [ ]:
def release_gpu(model=None):
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()


def benchmark_model(config):
    release_gpu()
    torch.cuda.reset_peak_memory_stats()
    print(f"Loading {config['model_id']} ...")
    model = SentenceTransformer(
        config['model_id'],
        device='cuda',
        model_kwargs={'torch_dtype': torch.float16},
        tokenizer_kwargs=config['tokenizer_kwargs'],
    )
    model.max_seq_length = 768

    torch.cuda.synchronize()
    started = time.perf_counter()
    document_vectors = model.encode(
        corpus_texts, batch_size=config['batch_size'],
        normalize_embeddings=True, convert_to_numpy=True,
        show_progress_bar=True,
    ).astype('float32', copy=False)
    torch.cuda.synchronize()
    document_seconds = time.perf_counter() - started

    query_kwargs = {'prompt_name': config['query_prompt_name']} if config['query_prompt_name'] else {}
    torch.cuda.synchronize()
    query_started = time.perf_counter()
    query_vectors = model.encode(
        questions, batch_size=config['batch_size'],
        normalize_embeddings=True, convert_to_numpy=True,
        show_progress_bar=False, **query_kwargs,
    ).astype('float32', copy=False)
    torch.cuda.synchronize()
    query_seconds = time.perf_counter() - query_started

    index = faiss.IndexFlatIP(document_vectors.shape[1])
    index.add(document_vectors)
    _, positions = index.search(query_vectors, 100)
    rankings = [[str(corpus_post_ids[position]) for position in row] for row in positions]
    metrics = score_post_rankings(eval_queries, rankings, recall_at=(5, 10), mrr_depth=10)

    result = {
        'name': config['name'],
        'model_id': config['model_id'],
        'dimensions': int(document_vectors.shape[1]),
        'chunks': len(document_vectors),
        **metrics,
        'documents_per_second': len(document_vectors) / document_seconds,
        'full_corpus_minutes': document_seconds / 60,
        'query_ms_each': query_seconds * 1000 / len(questions),
        'peak_vram_gib': torch.cuda.max_memory_allocated() / 1024**3,
        'flat_index_mib': document_vectors.nbytes / 1024**2,
    }
    del index, document_vectors, query_vectors, rankings
    release_gpu(model)
    return result


In [ ]:
if RESULTS_PATH.exists():
    checkpoint = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
    results = checkpoint.get('results', [])
    print(f'Resuming from {len(results)} completed model(s).')
else:
    results = []
completed_models = {result['name'] for result in results}
for model_config in MODEL_CONFIGS:
    if model_config['name'] in completed_models:
        print(f"Skipping completed model: {model_config['name']}")
        continue
    result = benchmark_model(model_config)
    results.append(result)
    RESULTS_PATH.write_text(json.dumps({
        'created_at': datetime.now(timezone.utc).isoformat(),
        'corpus_version': eval_dataset['corpus_version'],
        'label_status': eval_dataset['label_status'],
        'results': results,
    }, indent=2), encoding='utf-8')
    display(pd.DataFrame(results))
    print('Checkpointed:', RESULTS_PATH)


## 5. Read the trade-off before choosing

Do not choose the largest Recall number blindly. With 16 reviewed questions, tiny score differences can still be noise. Prefer the smaller/faster model when quality is effectively tied; accept Qwen's extra memory and index cost only when its answer-retrieval gain is meaningful.

The next production step after a defensible choice is: generate the selected vectors once, export `text + embedding + metadata` as Parquet, upload to S3, and run OpenSearch Vector Ingestion.


In [ ]:
comparison = pd.DataFrame(results).sort_values(['recall@10', 'mrr@10', 'documents_per_second'], ascending=[False, False, False])
display(comparison)
print('Answer-retrieval leader:', comparison.iloc[0]['name'])
print('Evidence scope: 16 human-reviewed answerable questions from corpus v1.')
